In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Load the provided CSV file
file_path = "./cifar100_results.csv"
df = pd.read_csv(file_path)

# Compute the difference between "original" and other transformation results
difference_df = df.copy()
for col in df.columns[3:]:  # Exclude "Metric" and "original"
    difference_df[col] = df["original"] - df[col]

# Drop the original column as we now have differences
difference_df = difference_df.drop(columns=["original"])

# Normalize the data
scaler = StandardScaler()
scaled_diff_data = scaler.fit_transform(difference_df.iloc[:, 2:])  # Exclude "Metric" column

# Determine the optimal number of clusters using the Elbow Method
inertia = []
K = range(1, 11)
for k in K:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=42)
    kmeans.fit(scaled_diff_data)
    inertia.append(kmeans.inertia_)

# Plot the Elbow Method graph
plt.figure(figsize=(8, 5))
plt.plot(K, inertia, 'bx-')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal k')
plt.show()

# Apply K-Means clustering on the difference data with the chosen k
optimal_k = 5  # 예시로 4를 사용, 엘보우 그래프를 보고 결정
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', n_init=10, max_iter=300, random_state=42)
difference_df["Cluster"] = kmeans.fit_predict(scaled_diff_data)

# Save the clustered data to a CSV file
output_file_path = "./cifar100_difference_clustered_results.csv"
difference_df.to_csv(output_file_path, index=False)

# Provide the download link
output_file_path

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 파일 경로 설정
sensitivity_file = './cifar100_difference_clustered_results.csv'

# CSV 파일 읽기
sensitivity_df = pd.read_csv(sensitivity_file)

# 민감도와 모델 성능 데이터 추출
sensitivity_methods = ['crop', 'flip', 'color_jitter', 'grayscale', 'translation', 'shearing', 'rotation', 'center_mask', 'noise_injection', 'kernel_filtering', 'random_erasing']

# 클러스터별로 데이터 그룹화
clusters = sensitivity_df['Cluster'].unique()

# 전체 데이터의 최소값과 최대값 계산
y_min = sensitivity_df[sensitivity_methods].min().min()
y_max = sensitivity_df[sensitivity_methods].max().max()

# 그래프 설정
fig, axes = plt.subplots(len(clusters), len(sensitivity_methods), figsize=(20, 5 * len(clusters)))

for i, cluster in enumerate(clusters):
    cluster_sensitivity_df = sensitivity_df[sensitivity_df['Cluster'] == cluster]
    
    for j, method in enumerate(sensitivity_methods):
        sns.boxplot(y=cluster_sensitivity_df[method], ax=axes[i, j])
        axes[i, j].set_title(f'Cluster {cluster} - {method}')
        axes[i, j].set_ylim(y_min, y_max)  # y축 범위 통일
        axes[i, j].set_xlabel('Sensitivity')
        axes[i, j].set_ylabel(method)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 파일 경로 설정
sensitivity_file = './cifar100_difference_clustered_results.csv'
performance_file = './dataAug/differences_model_results.csv'

# CSV 파일 읽기
sensitivity_df = pd.read_csv(sensitivity_file)
performance_df = pd.read_csv(performance_file)

# 민감도와 모델 성능 데이터 추출
sensitivity_methods = ['crop', 'flip', 'color_jitter', 'grayscale', 'translation', 'shearing', 'rotation', 'center_mask', 'noise_injection', 'kernel_filtering', 'random_erasing']
performance_methods = [f'{method}_accuracy_model_diff' for method in sensitivity_methods]

# 클러스터별로 데이터 그룹화
clusters = sensitivity_df['Cluster'].unique()

# 그래프 설정
fig, axes = plt.subplots(len(clusters), 2, figsize=(20, 5 * len(clusters)))

for i, cluster in enumerate(clusters):
    cluster_sensitivity_df = sensitivity_df[sensitivity_df['Cluster'] == cluster]
    cluster_performance_df = performance_df[performance_df['Cluster'] == cluster]

    # 상관관계 계산
    correlations = {}
    for sensitivity_method, performance_method in zip(sensitivity_methods, performance_methods):
        correlation = cluster_sensitivity_df[sensitivity_method].corr(cluster_performance_df[performance_method])
        correlations[sensitivity_method] = correlation

    # 상관관계 데이터프레임 생성
    correlation_df = pd.DataFrame(list(correlations.items()), columns=['Method', 'Correlation'])

    # 첫 번째 그래프: 민감도와 모델 성능 간의 상관관계
    sns.barplot(x='Method', y='Correlation', data=correlation_df, ax=axes[i, 0])
    axes[i, 0].set_title(f'Cluster {cluster} - Sensitivity vs Model Performance Correlation')
    axes[i, 0].set_xticklabels(axes[i, 0].get_xticklabels(), rotation=45)

    # 두 번째 그래프: 두 CSV 파일의 결과 비교
    sns.scatterplot(x=cluster_sensitivity_df['Class Index'], y=cluster_sensitivity_df['rotation'], label='Sensitivity', ax=axes[i, 1])
    sns.scatterplot(x=cluster_performance_df['Class Index'], y=cluster_performance_df['rotation_accuracy_model_diff'], label='Performance', ax=axes[i, 1])
    axes[i, 1].set_title(f'Cluster {cluster} - Sensitivity vs Performance for Rotation Transformation')
    axes[i, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load the provided CSV file
file_path = "./cifar100_difference_clustered_results.csv"
difference_df = pd.read_csv(file_path)

# Visualize the distribution of clusters
plt.figure(figsize=(8, 5))
sns.countplot(x=difference_df["Cluster"])
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.title("Cluster Distribution")
plt.grid(True)
plt.show()

difference_df.head()

# Extract class information from the 'Metric' column (if available)
if "Class Name" in difference_df.columns:
    difference_df["Class"] = difference_df["Class Name"]  # Add class names from original dataset

# Group by cluster and list the class names
cluster_class_mapping = difference_df.groupby("Cluster")["Class"].apply(list)

# Display the class names for each cluster
for cluster, class_list in cluster_class_mapping.items():
    print(f"🔹 **Cluster {cluster}** ({len(class_list)} classes):")
    print(", ".join(class_list))
    print("\n" + "-"*80 + "\n")


In [ ]:
# Visualize the average impact of transformations per cluster
cluster_means = difference_df.drop(columns=["Class Index"], errors="ignore").groupby("Cluster").mean(numeric_only=True).T

# Plot heatmap to see transformation differences by cluster
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)
plt.xlabel("Cluster")
plt.ylabel("Transformation Impact (Original - Transformed)")
plt.title("Transformation Impact by Cluster")
plt.show()

In [ ]:
# 클러스터별 민감도의 평균 계산
cluster_sensitivity_means = difference_df.groupby("Cluster").mean(numeric_only=True).drop(columns=["Class Index"])

# 클러스터별 민감도 시각화
for cluster in cluster_sensitivity_means.index:
    plt.figure(figsize=(10, 5))
    cluster_sensitivity_means.loc[cluster].plot(kind="bar")
    plt.title(f"Sensitivity per Method for Cluster {cluster}")
    plt.xlabel("Augmentation Method")
    plt.ylabel("Sensitivity Score")
    plt.xticks(rotation=45)
    plt.grid(axis="y")
    plt.show()


In [ ]:
# Scatter plot using first two principal components for visualization
# 문자열 데이터('Class Name' 열) 제거
if "Class Name" in difference_df.columns:
    difference_df = difference_df.drop(columns=["Class Index","Class Name"])

# 숫자형 데이터만 선택하여 스케일링
numeric_df = difference_df.select_dtypes(include=["number"])
scaled_diff_data = StandardScaler().fit_transform(numeric_df.drop(columns=["Cluster"]))  # 'Cluster' 제외 후 변환


pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_diff_data)
difference_df["PCA1"] = pca_result[:, 0]
difference_df["PCA2"] = pca_result[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(x="PCA1", y="PCA2", hue=difference_df["Cluster"], palette="tab10", data=difference_df)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("Clusters in PCA Space")
plt.legend(title="Cluster")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# CSV 파일 로드
file_path = "/hdd1/yebin/24-W-SimCLR/playground/results/dataAug/differences_model_results.csv"
difference_df = pd.read_csv(file_path)

# Total_Accuracy 행 제거
difference_df = difference_df[difference_df['Class Name'] != 'Total_Accuracy']

# 각 클래스별로 차이의 절대값 계산
numeric_columns = difference_df.columns[3:]
abs_difference_df = difference_df.copy()
abs_difference_df[numeric_columns] = abs(difference_df[numeric_columns])

# 각 클래스별로 가장 큰 차이를 보이는 방법론 찾기
difference_df['Max_Change_Method'] = abs_difference_df[numeric_columns].idxmax(axis=1)
difference_df['Max_Change_Value'] = difference_df.apply(lambda row: row[row['Max_Change_Method']], axis=1)

# 클러스터 순으로 정렬
difference_df = difference_df.sort_values(by='Cluster')

# 고유한 색상 팔레트 생성
unique_methods = difference_df['Max_Change_Method'].unique()
palette = sns.color_palette("hsv", len(unique_methods))
method_colors = dict(zip(unique_methods, palette))

# 클래스별로 50개씩 나누어 시각화
num_classes = len(difference_df)
num_chunks = (num_classes + 49) // 50  # 50개씩 나누기 위한 청크 수 계산

for i in range(num_chunks):
    start_idx = i * 50
    end_idx = min((i + 1) * 50, num_classes)
    chunk_df = difference_df.iloc[start_idx:end_idx]
    
    plt.figure(figsize=(20, 10))
    sns.barplot(x='Class Name', y='Max_Change_Value', hue='Max_Change_Method', data=chunk_df, dodge=False, palette=method_colors)
    for j in range(len(chunk_df)):
        plt.text(j, chunk_df['Max_Change_Value'].iloc[j], f"Cluster {chunk_df['Cluster'].iloc[j]}", ha='center', va='bottom', fontsize=9)
    plt.xlabel('Class Name')
    plt.ylabel('Maximum Performance Change')
    plt.title(f'Maximum Performance Change by Class (Classes {start_idx + 1} to {end_idx})')
    plt.xticks(rotation=90)
    plt.legend(title='Method')
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

# 파일 경로
cifar100_results_path = '/hdd1/yebin/24-W-SimCLR/playground/results/cifar100_results.csv'
model_results_path = '/hdd1/yebin/24-W-SimCLR/playground/results/dataAug/model_results.csv'
output_path = '/hdd1/yebin/24-W-SimCLR/playground/results/merged_results.csv'

# CSV 파일 읽기
cifar100_df = pd.read_csv(cifar100_results_path)
model_results_df = pd.read_csv(model_results_path)

# 공통된 열을 기준으로 병합
merged_df = pd.merge(cifar100_df, model_results_df, on=['Class Index', 'Class Name'], suffixes=('_cifar100', '_model'))

# 병합된 데이터프레임을 CSV 파일로 저장
merged_df.to_csv(output_path, index=False)

print(f"Merged data has been successfully written to {output_path}")

Merged data has been successfully written to /hdd1/yebin/24-W-SimCLR/playground/results/merged_results.csv
